# Model Development

The goal is to predict students' O-level Mathematics scores (`final_test`) so that the school can identify weaker students before the examination. This notebook:

1. Splits the data into training, validation and test sets
2. Builds a preprocessing pipeline that is fitted on the training data only
3. Compares linear models and non-linear models using cross-validation
4. Tunes the most promising models
5. Selects a final model using a clear rule
6. Refits the chosen model on the training and validation data, and evaluates it once on the test set
7. Saves the final model and its results

In [41]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Settings used throughout the notebook
RANDOM_STATE = 42
TARGET = 'final_test'
WEAK_THRESHOLD = 50  # Students predicted to score below this are flagged as weaker students

## Data Splitting

In [24]:
df = pd.read_csv("data/cleaned_student_records.csv")
print('Dataset shape:', df.shape)
df.head()

Dataset shape: (14559, 10)


,number_of_siblings,direct_admission,CCA,learning_style,tuition,final_test,hours_per_week,attendance_rate,class_size,sleep_hours
0,0,1,Sports,1,0,69.0,10.0,91.0,16.0,8.0
1,2,0,Sports,0,0,47.0,7.0,94.0,23.0,8.0
2,0,1,No CCA,1,0,85.0,8.0,92.0,16.0,8.0
3,1,0,Clubs,0,1,64.0,18.0,NaN,22.0,8.0
4,0,0,Sports,0,0,66.0,7.0,95.0,27.0,8.0


In [25]:
# Separate the data into features and target
X = df.drop(columns=TARGET)
y = df[TARGET]

In [26]:
from sklearn.model_selection import train_test_split

# Split the data into training (80%) and a temporary set (20%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

# Split the temporary set equally into validation (10%) and test (10%) sets
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE)

# Verify the split sizes
print('Training set:  ', X_train.shape, y_train.shape)
print('Validation set:', X_val.shape, y_val.shape)
print('Test set:      ', X_test.shape, y_test.shape)

# Verify that no record appears in more than one set
overlap = set(X_train.index) & set(X_val.index) | set(X_train.index) & set(X_test.index) | set(X_val.index) & set(X_test.index)
print('Rows shared between sets:', len(overlap))

Training set:   (11647, 9) (11647,)
Validation set: (1456, 9) (1456,)
Test set:       (1456, 9) (1456,)
Rows shared between sets: 0


The data is split into 80% training, 10% validation and 10% test. The training set is used to fit and tune the models, the validation set is used to compare them, and the test set is kept untouched until the very end so that it gives a fair estimate of performance on unseen students.

## Preprocessing Pipeline

### Imputing Missing Values

`attendance_rate` still has missing values after data cleaning. They are filled with the median using `SimpleImputer`, which is the first step of the numerical pipeline below.

As the pipeline is only fitted on the training data, the median is calculated from the training set only, and that same value is used to fill the validation and test sets. This avoids data leakage. During cross-validation, the pipeline is refitted in each fold, so the median is also recalculated from each fold's training data.

### Feature Scaling

Numerical features are standardised because they have very different ranges, such as `attendance_rate` (40 to 100) and `number_of_siblings` (0 to 2). Without scaling, features with larger values could have a greater influence on linear models and on the penalty used by Ridge and Lasso. Tree-based models are not affected by scaling, but keeping it in one shared pipeline makes the comparison between models consistent.

In [27]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Numerical features to be imputed and standardised
numerical_features = ['number_of_siblings', 'hours_per_week', 'attendance_rate', 'class_size', 'sleep_hours']

# Missing values are filled with the training set median before standardising
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

### Feature Encoding

`CCA` is one-hot encoded as it has four categories with no natural order. `handle_unknown='ignore'` means the pipeline will not fail if a new category appears in future data. `direct_admission`, `learning_style` and `tuition` were already converted to 1 and 0 during data cleaning, so they are passed through unchanged.

In [28]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Nominal features to be one-hot encoded
nominal_features = ['CCA']

# Binary features already encoded as 1 and 0
passthrough_features = ['direct_admission', 'learning_style', 'tuition']

nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine the transformers into a single preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_features),
    ('nom', nominal_transformer, nominal_features),
    ('pass', 'passthrough', passthrough_features) # Pass through the binary features without transformation
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

## Helper Functions

The same steps are repeated for every model: attaching the preprocessor, and calculating MAE, RMSE and R². To avoid repeating the same code for each model, these are wrapped in two small functions.

In [29]:
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

def build_pipeline(model):
    """Attach a fresh copy of the preprocessor to a model."""
    return Pipeline(steps=[
        ('preprocessor', clone(preprocessor)),
        ('regressor', model)
    ])

def evaluate(model, X, y, name):
    """Return MAE, RMSE and R² for a fitted model on the given data."""
    y_pred = model.predict(X)
    return {
        'model': name,
        'MAE': mean_absolute_error(y, y_pred),
        'MSE': mean_squared_error(y, y_pred),
        'RMSE': root_mean_squared_error(y, y_pred),
        'R²': r2_score(y, y_pred)
    }

## Baseline Model Comparison

Before tuning, several models are compared with their default settings using 5-fold cross-validation on the training set. Cross-validation gives a more reliable comparison than a single validation split, as each model is trained and tested five times on different parts of the data. The mean shows the typical performance, and the standard deviation shows how stable it is.

### Choice of Models

As `final_test` is a continuous score, regression models are used. They were chosen to move from simple and interpretable to more complex, so the extra complexity is only kept if it improves predictions.

- Linear Regression is the simplest and most interpretable model, as each coefficient shows how a feature raises or lowers the predicted score.
- Ridge and Lasso test whether regularisation helps, as some features are related (e.g. `attendance_rate` and `sleep_hours`, VIF around 4.7). Lasso can also shrink less useful features to zero.
- Random Forest captures threshold effects and interactions that linear models cannot, such as students below about 90% attendance all scoring 50 or below.
- Gradient Boosting is a stronger non-linear model that often performs best on tabular data, used to check whether extra complexity beyond Random Forest is worthwhile.

In [31]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

# Use the same 5 folds for every model so the comparison is fair
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1),
    'Lasso (alpha=1)': Lasso(alpha=1),
    'Random Forest': RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': HistGradientBoostingRegressor(random_state=RANDOM_STATE)
}

baseline_results = []
for name, model in baseline_models.items():
    scores = cross_validate(build_pipeline(model), X_train, y_train, cv=cv,
                            scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'])
    baseline_results.append({
        'model': name,
        'CV MAE': -scores['test_neg_mean_absolute_error'].mean(),
        'CV RMSE': -scores['test_neg_root_mean_squared_error'].mean(),
        'CV R² (mean)': scores['test_r2'].mean(),
        'CV R² (std)': scores['test_r2'].std()
    })

baseline_df = pd.DataFrame(baseline_results).sort_values('CV R² (mean)', ascending=False).round(4)
baseline_df

,model,CV MAE,CV RMSE,CV R² (mean),CV R² (std)
4,Gradient Boosting,3.8353,5.3829,0.8517,0.0096
3,Random Forest,3.9410,5.7593,0.8303,0.0089
1,Ridge (alpha=1),7.2732,9.0945,0.5770,0.0068
0,Linear Regression,7.2731,9.0945,0.5770,0.0068
2,Lasso (alpha=1),7.9247,9.7951,0.5094,0.0045


There is a large gap between the linear and non-linear models. Linear Regression and Ridge reach a cross-validated R² of about 0.58, with an average error of 7.3 marks. 

Random Forest and Gradient Boosting reach 0.83 and 0.85, with an average error below 4 marks, roughly half that of the linear models. This supports the EDA finding that some relationships, such as the attendance threshold, are not linear. The small standard deviations (around 0.01) show that these results are stable across all five folds.

Default Lasso performs worse than Linear Regression (0.51), because its default penalty (`alpha=1`) shrinks useful coefficients too much.

## Hyperparameter Tuning

The four most relevant models are tuned using 5-fold cross-validation on the training set, with R² as the scoring metric:

- **Ridge and Lasso** each have one main hyperparameter, `alpha`, so `GridSearchCV` is used to test every value on a log scale. The range goes down to 0.0001, so that the best value is not stuck at the edge of the search range. `fit_intercept` is not tuned and is left at its default of `True`. The intercept is the model's starting prediction for an average student, which should be close to the average score of about 67. Setting it to `False` would force this starting prediction to zero, which does not reflect the data, so it is not a meaningful option to test.
- **Random Forest and Gradient Boosting** have several hyperparameters, and testing every combination would take too long. `RandomizedSearchCV` is used instead, which tests a fixed number of random combinations drawn from each range. This usually finds a near-best combination in far less time.

In [32]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

alphas = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]

# Ridge: grid search over alpha
ridge_search = GridSearchCV(build_pipeline(Ridge()), {'regressor__alpha': alphas},
                            cv=cv, scoring='r2', n_jobs=-1)
ridge_search.fit(X_train, y_train)

# Lasso: grid search over alpha
lasso_search = GridSearchCV(build_pipeline(Lasso(max_iter=10000)), {'regressor__alpha': alphas},
                            cv=cv, scoring='r2', n_jobs=-1)
lasso_search.fit(X_train, y_train)

print('Best Ridge alpha:', ridge_search.best_params_, '| CV R²:', round(ridge_search.best_score_, 4))
print('Best Lasso alpha:', lasso_search.best_params_, '| CV R²:', round(lasso_search.best_score_, 4))

Best Ridge alpha: {'regressor__alpha': 10} | CV R²: 0.577
Best Lasso alpha: {'regressor__alpha': 0.01} | CV R²: 0.577


In [34]:
# Random Forest: randomised search over several hyperparameters
rf_params = {
    'regressor__n_estimators': randint(100, 400),
    'regressor__max_depth': [None, 10, 15, 20],
    'regressor__min_samples_leaf': randint(1, 10),
    'regressor__max_features': [0.5, 0.7, 1.0]
}
rf_search = RandomizedSearchCV(build_pipeline(RandomForestRegressor(random_state=RANDOM_STATE)), rf_params,
                               n_iter=15, cv=cv, scoring='r2', random_state=RANDOM_STATE, n_jobs=-1)
rf_search.fit(X_train, y_train)

# Gradient Boosting: randomised search over several hyperparameters
gb_params = {
    'regressor__learning_rate': loguniform(0.02, 0.3),
    'regressor__max_iter': randint(100, 500),
    'regressor__max_depth': [None, 3, 5, 8],
    'regressor__min_samples_leaf': randint(10, 50),
    'regressor__l2_regularization': uniform(0, 1)
}
gb_search = RandomizedSearchCV(build_pipeline(HistGradientBoostingRegressor(random_state=RANDOM_STATE)), gb_params,
                               n_iter=20, cv=cv, scoring='r2', random_state=RANDOM_STATE, n_jobs=-1)
gb_search.fit(X_train, y_train)

print('Best Random Forest parameters:', rf_search.best_params_, '| CV R²:', round(rf_search.best_score_, 4))
print('Best Gradient Boosting parameters:', gb_search.best_params_, '| CV R²:', round(gb_search.best_score_, 4))

Best Random Forest parameters: {'regressor__max_depth': 15, 'regressor__max_features': 0.7, 'regressor__min_samples_leaf': 3, 'regressor__n_estimators': 154} | CV R²: 0.8488
Best Gradient Boosting parameters: {'regressor__l2_regularization': np.float64(0.9093204020787821), 'regressor__learning_rate': np.float64(0.04030685101059539), 'regressor__max_depth': 8, 'regressor__max_iter': 487, 'regressor__min_samples_leaf': 11} | CV R²: 0.8532


The best `alpha` for Ridge is 10 and for Lasso is 0.01. Both are inside the search range rather than at its edge, so the range was wide enough. Tuning did not improve either model beyond Linear Regression (all around 0.577), as the dataset has far more students than features, so the linear models are not overfitting and regularisation has little to correct.

Tuning improved Random Forest from 0.830 to 0.849, mainly by limiting tree depth and requiring a minimum number of students in each leaf, which reduces overfitting. Gradient Boosting improved slightly from 0.852 to 0.853, suggesting its default settings were already close to optimal.

## Model Comparison and Selection

The tuned models are compared using two sources of evidence:

1. **Cross-validation on the training set**, which shows typical performance and how stable it is across folds.
2. **The validation set**, which shows performance on students the models have never been trained on.

Linear Regression is included as the untuned reference. The tuned models are taken directly from `best_estimator_`, rather than rebuilt with hard-coded values, so the results always match what the search found.

In [36]:
searches = {
    'Ridge (tuned)': ridge_search,
    'Lasso (tuned)': lasso_search,
    'Random Forest (tuned)': rf_search,
    'Gradient Boosting (tuned)': gb_search
}

# Collect the best model and its cross-validation score from each search
candidates = {}
for name, search in searches.items():
    best_idx = search.best_index_
    candidates[name] = {
        'model': search.best_estimator_,
        'cv_mean': search.cv_results_['mean_test_score'][best_idx],
        'cv_std': search.cv_results_['std_test_score'][best_idx]
    }

# Add untuned Linear Regression as a reference
lr_scores = cross_validate(build_pipeline(LinearRegression()), X_train, y_train, cv=cv, scoring='r2')['test_score']
candidates['Linear Regression'] = {
    'model': build_pipeline(LinearRegression()).fit(X_train, y_train),
    'cv_mean': lr_scores.mean(),
    'cv_std': lr_scores.std()
}

# Evaluate each candidate on the validation set
comparison = []
for name, c in candidates.items():
    val_metrics = evaluate(c['model'], X_val, y_val, name)
    comparison.append({**val_metrics, 'CV R² (mean)': c['cv_mean'], 'CV R² (std)': c['cv_std']})

comparison_df = (pd.DataFrame(comparison)
                 .sort_values('CV R² (mean)', ascending=False)
                 .round(4))
comparison_df

,model,MAE,MSE,RMSE,R²,CV R² (mean),CV R² (std)
3,Gradient Boosting (tuned),3.8072,29.7042,5.4502,0.8464,0.8532,0.0106
2,Random Forest (tuned),3.7307,29.9823,5.4756,0.8450,0.8488,0.0099
1,Lasso (tuned),7.2754,82.0692,9.0592,0.5756,0.5770,0.0067
0,Ridge (tuned),7.2743,82.0592,9.0587,0.5757,0.5770,0.0067
4,Linear Regression,7.2749,82.0724,9.0594,0.5756,0.5770,0.0068


### Selection Rule

To make the choice clear and repeatable, the final model is selected using this rule:

1. Pick the model with the highest mean cross-validation R².
2. If another model's score is within one standard deviation of the best, treat them as tied, and choose the **simpler** model, as it is easier to explain and less likely to overfit.

The models are ranked from simplest to most complex as: Linear Regression, Ridge, Lasso, Random Forest, Gradient Boosting.

In [37]:
complexity_order = ['Linear Regression', 'Ridge (tuned)', 'Lasso (tuned)',
                    'Random Forest (tuned)', 'Gradient Boosting (tuned)']

best_name = max(candidates, key=lambda n: candidates[n]['cv_mean'])
best_mean, best_std = candidates[best_name]['cv_mean'], candidates[best_name]['cv_std']

# Models whose score is within one standard deviation of the best are treated as tied
tied = [n for n in candidates if best_mean - candidates[n]['cv_mean'] <= best_std]

# Among tied models, choose the simplest
selected_name = min(tied, key=complexity_order.index)

print(f'Highest CV R²: {best_name} ({best_mean:.4f} ± {best_std:.4f})')
print('Models tied within one standard deviation:', tied)
print('Selected model:', selected_name)

Highest CV R²: Gradient Boosting (tuned) (0.8532 ± 0.0106)
Models tied within one standard deviation: ['Random Forest (tuned)', 'Gradient Boosting (tuned)']
Selected model: Random Forest (tuned)


Gradient Boosting has the highest mean cross-validation R² (0.853), but Random Forest (0.849) is within one standard deviation of it, so the two are effectively tied. Following the selection rule, the simpler of the two, **Random Forest**, is selected.

On the validation set, the two models also perform almost the same. Random Forest has a slightly lower MAE (3.73 compared to 3.81), while Gradient Boosting has a slightly higher R² (0.846 compared to 0.845). Both explain far more of the variation in scores than the linear models (about 0.58).

## Final Model Evaluation

Before the final evaluation, the selected model is refitted on the combined training and validation sets, using the tuned hyperparameters. This lets the model learn from more data. The test set has not been used at any stage so far, so it gives a fair estimate of how the model would perform on new students. It is used only once.

In [38]:
# Refit the selected model on the combined training and validation data
final_model = clone(candidates[selected_name]['model'])
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])
final_model.fit(X_train_full, y_train_full)

# Evaluate once on the untouched test set
test_metrics = evaluate(final_model, X_test, y_test, selected_name)
y_test_pred = final_model.predict(X_test)

val_row = comparison_df.loc[comparison_df['model'] == selected_name].iloc[0]
pd.DataFrame({
    'Validation': [val_row['MAE'], val_row['MSE'], val_row['RMSE'], val_row['R²']],
    'Test': [test_metrics['MAE'], test_metrics['MSE'], test_metrics['RMSE'], test_metrics['R²']]
}, index=['MAE', 'MSE', 'RMSE', 'R²']).round(4)

,Validation,Test
MAE,3.7307,3.7828
MSE,29.9823,29.0215
RMSE,5.4756,5.3872
R²,0.8450,0.8508


After refitting on the combined training and validation data, the Random Forest model achieves an R² of 0.85 on the test set, with an average error of 3.78 marks. These results are very close to the validation results (R² of 0.845, MAE of 3.73), which suggests that the model generalises well to students it has never seen, rather than only performing well on the data used during model development.

## Saving the Model and Results

The final pipeline, a record of the selected model and its metrics, the full model comparison and the test set predictions are saved to an `outputs` folder. This keeps the results available after the notebook is closed, and makes it possible to trace which model was chosen and why.

In [39]:
import json
import joblib
from pathlib import Path

output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

# Save the full pipeline (preprocessing and model together)
joblib.dump(final_model, output_dir / 'final_model.joblib')

# Save a record of the selected model, its hyperparameters and its metrics
with open(output_dir / 'final_metrics.json', 'w') as f:
    json.dump({
        'selected_model': selected_name,
        'hyperparameters': {k: str(v) for k, v in searches[selected_name].best_params_.items()} if selected_name in searches else {},
        'cv_r2_mean': round(float(candidates[selected_name]['cv_mean']), 4),
        'validation_metrics': {k: round(float(val_row[k]), 4) for k in ['MAE', 'MSE', 'RMSE', 'R²']},
        'test_metrics': {k: round(float(v), 4) for k, v in test_metrics.items() if k != 'model'}
    }, f, indent=2)

# Save the model comparison and the test set predictions
comparison_df.to_csv(output_dir / 'model_comparison.csv', index=False)
pd.DataFrame({'actual': y_test, 'predicted': y_test_pred}).to_csv(output_dir / 'test_predictions.csv')

print("Saved files:", sorted(p.name for p in output_dir.iterdir()))

Saved files: ['final_metrics.json', 'final_model.joblib', 'model_comparison.csv', 'test_predictions.csv']


## Predicting for New Students

The saved pipeline can be loaded later and used to predict scores for new students without retraining. As the preprocessing steps are saved inside the pipeline, new records only need to be in the same format as the cleaned data.

In [40]:
# Load the saved pipeline
loaded_model = joblib.load(output_dir / 'final_model.joblib')

# Confirm the loaded model gives the same predictions as before saving
print("Loaded model matches:", (loaded_model.predict(X_test) == y_test_pred).all())

# Example: predict scores for two new students
new_students = pd.DataFrame([
    {'number_of_siblings': 0, 'direct_admission': 1, 'CCA': 'Sports', 'learning_style': 1, 'tuition': 1,
     'hours_per_week': 10, 'attendance_rate': 95, 'class_size': 20, 'sleep_hours': 8},
    {'number_of_siblings': 2, 'direct_admission': 0, 'CCA': 'Clubs', 'learning_style': 0, 'tuition': 0,
     'hours_per_week': 5, 'attendance_rate': 80, 'class_size': 30, 'sleep_hours': 5}
])
new_students['predicted_final_test'] = loaded_model.predict(new_students).round(1)
new_students

Loaded model matches: True


,number_of_siblings,direct_admission,CCA,learning_style,tuition,hours_per_week,attendance_rate,class_size,sleep_hours,predicted_final_test
0,0,1,Sports,1,1,10,95,20,8,82.6
1,2,0,Clubs,0,0,5,80,30,5,47.4


The loaded model gives exactly the same predictions as before it was saved, confirming the pipeline was saved correctly. It also predicts sensible scores for new students: a student with high attendance, full sleep and a small class is predicted to score 82.7, while a student with lower attendance, less sleep and a larger class is predicted to score 48.0.

## Conclusion

A Random Forest model was selected to predict students' O-level Mathematics scores, to help the school identify weaker students before the examination. On the test set, it achieved an R² of 0.848 and an average error of 3.8 marks.

This is a large improvement over the linear models, which achieved an R² of about 0.58 and an average error of about 7.3 marks. The improvement comes from the model's ability to capture non-linear patterns and interactions between features, which the EDA had already suggested were present.